Для подтверждения целесообразности использования нейронной сети обученной на кадре до и после демонструется обучение только на кадре после

In [7]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms.functional as TF

from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.models import convnext_small, ConvNeXt_Small_Weights

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

In [8]:
CSV_PATH = Path(r"D:\diplom\dataset_before_after\cropped_pairs\before_after_pairs_cropped.csv")
WEIGHTS_DIR = Path(r"D:\diplom\weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

In [9]:
IMAGE_COLUMN = "after_path"
MODEL_NAME = "convnext_small_after_only"
BEST_WEIGHTS_PATH = WEIGHTS_DIR / "convnext_small_after_only_cropped.pth"

In [10]:
IMG_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 80
WARMUP_EPOCHS = 5

HEAD_LR_WARMUP = 1e-3
BACKBONE_LR = 3e-5
HEAD_LR_FINETUNE = 2e-4
WEIGHT_DECAY = 1e-2

VAL_SIZE = 0.2
RANDOM_STATE = 42
NUM_WORKERS = 0

PATIENCE = 12
MIN_DELTA = 2e-3
GRAD_CLIP_NORM = 1.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(RANDOM_STATE)

In [12]:
def normalize_binary_value(x):
    if pd.isna(x):
        return None

    s = str(x).strip().lower().replace(",", ".")
    mapping = {
        "0": 0,
        "0.0": 0,
        "false": 0,
        "no": 0,
        "нет": 0,
        "1": 1,
        "1.0": 1,
        "true": 1,
        "yes": 1,
        "да": 1,
    }
    return mapping.get(s, None)


def load_dataframe(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"Не найден CSV: {csv_path}")

    df = pd.read_csv(csv_path, encoding="utf-8-sig")

    required_cols = ["video_file", "before_path", "after_path", "result"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"В CSV отсутствуют столбцы: {missing}")

    df = df.copy()
    df["video_file"] = df["video_file"].astype(str).str.strip()
    df["before_path"] = df["before_path"].astype(str).str.strip()
    df["after_path"] = df["after_path"].astype(str).str.strip()

    df = df[(df["before_path"] != "") & (df["after_path"] != "")]
    df = df[df["before_path"].apply(lambda p: Path(p).exists())]
    df = df[df["after_path"].apply(lambda p: Path(p).exists())]

    df["label"] = df["result"].apply(normalize_binary_value)
    df = df[df["label"].notna()].copy()
    df["label"] = df["label"].astype(int)

    unique_labels = sorted(df["label"].unique().tolist())
    if unique_labels != [0, 1]:
        raise ValueError(f"Ожидались бинарные метки 0/1. Найдено: {unique_labels}")

    if len(df) == 0:
        raise ValueError("После фильтрации не осталось валидных строк.")

    return df.reset_index(drop=True)


df = load_dataframe(CSV_PATH)

print(f"Модель: {MODEL_NAME}")
print(f"Используемая колонка: {IMAGE_COLUMN}")
print(f"Всего валидных объектов: {len(df)}")
print("Распределение классов:")
print(df["label"].value_counts().sort_index())

Модель: convnext_small_after_only
Используемая колонка: after_path
Всего валидных объектов: 339
Распределение классов:
label
0     67
1    272
Name: count, dtype: int64


In [13]:
groups = df["video_file"].values
y = df["label"].values

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE
)

train_idx, val_idx = next(gss.split(df, y=y, groups=groups))
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

print(f"\nTrain samples: {len(train_df)}")
print(f"Val samples:   {len(val_df)}")
print(f"Train videos:  {train_df['video_file'].nunique()}")
print(f"Val videos:    {val_df['video_file'].nunique()}")


Train samples: 271
Val samples:   68
Train videos:  271
Val videos:    68


In [14]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class TrainTransform:
    def __init__(self, img_size=224):
        self.img_size = img_size
        self.resize_size = 256

    def __call__(self, img):
        img = TF.resize(
            img, [self.resize_size, self.resize_size],
            interpolation=InterpolationMode.BILINEAR
        )

        i, j, h, w = transforms.RandomResizedCrop.get_params(
            img, scale=(0.88, 1.0), ratio=(0.95, 1.05)
        )
        img = TF.resized_crop(
            img, i, j, h, w,
            [self.img_size, self.img_size],
            interpolation=InterpolationMode.BILINEAR
        )

        if random.random() < 0.5:
            img = TF.hflip(img)

        angle = random.uniform(-7.0, 7.0)
        img = TF.rotate(
            img, angle,
            interpolation=InterpolationMode.BILINEAR,
            fill=0
        )

        brightness = random.uniform(0.95, 1.05)
        contrast = random.uniform(0.95, 1.05)
        img = TF.adjust_brightness(img, brightness)
        img = TF.adjust_contrast(img, contrast)

        x = TF.to_tensor(img)
        x = TF.normalize(x, mean=IMAGENET_MEAN, std=IMAGENET_STD)
        return x


class ValTransform:
    def __init__(self, img_size=224):
        self.img_size = img_size

    def __call__(self, img):
        img = TF.resize(
            img, [self.img_size, self.img_size],
            interpolation=InterpolationMode.BILINEAR
        )
        x = TF.to_tensor(img)
        x = TF.normalize(x, mean=IMAGENET_MEAN, std=IMAGENET_STD)
        return x


train_transform = TrainTransform(img_size=IMG_SIZE)
val_transform = ValTransform(img_size=IMG_SIZE)

In [15]:
class SingleFrameDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, image_column: str, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_column = image_column
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row[self.image_column]).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)

        label = torch.tensor(row["label"], dtype=torch.float32)
        return img, label


train_ds = SingleFrameDataset(train_df, image_column=IMAGE_COLUMN, transform=train_transform)
val_ds = SingleFrameDataset(val_df, image_column=IMAGE_COLUMN, transform=val_transform)

In [16]:
class_counts = train_df["label"].value_counts().to_dict()
sample_weights = train_df["label"].map(lambda x: 1.0 / class_counts[x]).values
sample_weights = torch.DoubleTensor(sample_weights)

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

C:\Users\aladina\AppData\Local\Temp\ipykernel_16140\1191972072.py:3: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  sample_weights = torch.DoubleTensor(sample_weights)


In [17]:
class SingleImageConvNeXtSmall(nn.Module):
    def __init__(self, dropout=0.40):
        super().__init__()

        backbone = convnext_small(weights=ConvNeXt_Small_Weights.DEFAULT)

        feat_dim = backbone.classifier[2].in_features
        backbone.classifier[2] = nn.Identity()
        self.backbone = backbone

        self.head = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(128, 1),
        )

    def forward(self, x):
        feats = self.backbone(x)
        logits = self.head(feats).squeeze(1)
        return logits


model = SingleImageConvNeXtSmall().to(DEVICE)

In [18]:
def set_backbone_trainable(model, trainable: bool):
    for p in model.backbone.parameters():
        p.requires_grad = trainable


def build_optimizer_warmup(model):
    return torch.optim.AdamW(
        model.head.parameters(),
        lr=HEAD_LR_WARMUP,
        weight_decay=WEIGHT_DECAY
    )


def build_optimizer_finetune(model):
    return torch.optim.AdamW(
        [
            {"params": model.backbone.parameters(), "lr": BACKBONE_LR},
            {"params": model.head.parameters(), "lr": HEAD_LR_FINETUNE},
        ],
        weight_decay=WEIGHT_DECAY
    )


def build_scheduler(optimizer):
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )


def compute_metrics(y_true, logits):
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs >= 0.5).astype(np.int64)

    acc = accuracy_score(y_true, preds)
    f1 = f1_score(y_true, preds, zero_division=0)

    if len(np.unique(y_true)) == 2:
        auc = roc_auc_score(y_true, probs)
    else:
        auc = float("nan")

    return acc, f1, auc


def run_epoch(loader, model, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_labels = []
    all_logits = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_logits.extend(logits.detach().cpu().numpy().tolist())

    epoch_loss = total_loss / len(loader.dataset)
    acc, f1, auc = compute_metrics(np.array(all_labels), np.array(all_logits))
    return epoch_loss, acc, f1, auc


class EarlyStopping:
    def __init__(self, patience=12, min_delta=0.002):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = None
        self.counter = 0
        self.should_stop = False

    def reset(self):
        self.best_score = None
        self.counter = 0
        self.should_stop = False

    def step(self, score):
        if self.best_score is None:
            self.best_score = score
            self.counter = 0
            return True

        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter = 0
            return True

        self.counter += 1
        if self.counter >= self.patience:
            self.should_stop = True
        return False


criterion = nn.BCEWithLogitsLoss()

In [19]:
set_backbone_trainable(model, False)
optimizer = build_optimizer_warmup(model)
scheduler = build_scheduler(optimizer)
early_stopper = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA)

for epoch in range(1, EPOCHS + 1):
    if epoch == WARMUP_EPOCHS + 1:
        print("\n--- Разморозка backbone и переход к finetune ---")
        set_backbone_trainable(model, True)
        optimizer = build_optimizer_finetune(model)
        scheduler = build_scheduler(optimizer)
        early_stopper.reset()

    train_loss, train_acc, train_f1, train_auc = run_epoch(
        train_loader, model, criterion, optimizer=optimizer
    )

    val_loss, val_acc, val_f1, val_auc = run_epoch(
        val_loader, model, criterion, optimizer=None
    )

    score = val_auc if not np.isnan(val_auc) else val_f1
    scheduler.step(score)

    if len(optimizer.param_groups) == 1:
        lr_info = f"head_lr={optimizer.param_groups[0]['lr']:.6f}"
    else:
        lr_info = (
            f"backbone_lr={optimizer.param_groups[0]['lr']:.6f} "
            f"head_lr={optimizer.param_groups[1]['lr']:.6f}"
        )

    print(
        f"[{epoch:02d}/{EPOCHS}] "
        f"{lr_info} "
        f"train_loss={train_loss:.4f} "
        f"train_acc={train_acc:.4f} "
        f"train_f1={train_f1:.4f} "
        f"train_auc={train_auc:.4f} | "
        f"val_loss={val_loss:.4f} "
        f"val_acc={val_acc:.4f} "
        f"val_f1={val_f1:.4f} "
        f"val_auc={val_auc:.4f}"
    )

    improved = early_stopper.step(score)
    if improved:
        torch.save(model.state_dict(), BEST_WEIGHTS_PATH)
        print("  -> сохранены лучшие веса")
    else:
        print(f"  -> без улучшения: {early_stopper.counter}/{PATIENCE}")

    if early_stopper.should_stop:
        print("\nEarly stopping: обучение остановлено.")
        break

[01/80] head_lr=0.001000 train_loss=0.7086 train_acc=0.5166 train_f1=0.6158 train_auc=0.4812 | val_loss=0.6695 val_acc=0.6618 val_f1=0.7723 val_auc=0.5571
  -> сохранены лучшие веса
[02/80] head_lr=0.001000 train_loss=0.7015 train_acc=0.5351 train_f1=0.4793 train_auc=0.5586 | val_loss=0.7030 val_acc=0.5000 val_f1=0.5641 val_auc=0.5559
  -> без улучшения: 1/12
[03/80] head_lr=0.001000 train_loss=0.6866 train_acc=0.5609 train_f1=0.2788 train_auc=0.5462 | val_loss=0.7334 val_acc=0.5441 val_f1=0.6173 val_auc=0.5409
  -> без улучшения: 2/12
[04/80] head_lr=0.000500 train_loss=0.6737 train_acc=0.5978 train_f1=0.6121 train_auc=0.6225 | val_loss=0.7709 val_acc=0.4412 val_f1=0.4865 val_auc=0.5456
  -> без улучшения: 3/12
[05/80] head_lr=0.000500 train_loss=0.6822 train_acc=0.5720 train_f1=0.5538 train_auc=0.6222 | val_loss=0.6250 val_acc=0.6618 val_f1=0.7850 val_auc=0.5156
  -> без улучшения: 4/12

--- Разморозка backbone и переход к finetune ---
[06/80] backbone_lr=0.000030 head_lr=0.000200 tr

In [20]:
model.load_state_dict(torch.load(BEST_WEIGHTS_PATH, map_location=DEVICE))

final_val_loss, final_val_acc, final_val_f1, final_val_auc = run_epoch(
    val_loader, model, criterion, optimizer=None
)

print("\nЛучшие сохранённые веса:")
print(BEST_WEIGHTS_PATH)

print("\nИтоговые метрики на validation:")
print(f"val_loss = {final_val_loss:.4f}")
print(f"val_acc  = {final_val_acc:.4f}")
print(f"val_f1   = {final_val_f1:.4f}")
print(f"val_auc  = {final_val_auc:.4f}")


Лучшие сохранённые веса:
D:\diplom\weights\convnext_small_after_only_cropped.pth

Итоговые метрики на validation:
val_loss = 0.6282
val_acc  = 0.6618
val_f1   = 0.7723
val_auc  = 0.5779
